# 📉 Semana 4: Regresión Lineal Simple y Múltiple

## Objetivos
- Comprender los fundamentos y supuestos de la regresión lineal
- Implementar Regresión Lineal Simple con scikit-learn
- Evaluar modelos con R², MSE, RMSE, MAE
- Construir modelos de Regresión Lineal Múltiple
- Validar los supuestos del modelo
- Desarrollar un pipeline completo desde la carga hasta el modelo final

## Dataset: Water Quality (Calidad del Agua)
Aplicaremos todo lo aprendido en las semanas anteriores para construir modelos predictivos.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("✅ Librerías importadas")

In [ ]:
# Carga y preparación del dataset (pipeline de semanas anteriores)
url = "https://raw.githubusercontent.com/jaquimbayoc7/material-fundamentos_datos/main/data/water_potability.csv"
df = pd.read_csv(url)

print(f"Dataset original: {df.shape}")
print(f"Valores nulos: {df.isnull().sum().sum()}")

# Imputar valores faltantes con la mediana
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Eliminar duplicados
df = df.drop_duplicates().reset_index(drop=True)

cols_numericas = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'Potability']

print(f"\n✅ Dataset limpio: {df.shape}")
print(f"Sin nulos: {df.isnull().sum().sum() == 0}")
df.head()

---
# 1. Fundamentos de la Regresión Lineal

## Concepto

La **regresión lineal** modela la relación entre una variable dependiente (Y) y una o más variables independientes (X) como una función lineal:

### Regresión Simple:
$$\hat{Y} = \beta_0 + \beta_1 X$$

### Regresión Múltiple:
$$\hat{Y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_n X_n$$

Donde:
- $\hat{Y}$: Valor predicho
- $\beta_0$: Intercepto (valor de Y cuando todas las X = 0)
- $\beta_i$: Coeficiente de la variable $X_i$ (pendiente)

## Supuestos de la Regresión Lineal

| Supuesto | Descripción | Prueba |
|----------|-------------|--------|
| **Linealidad** | Relación lineal entre X e Y | Gráfico de dispersión |
| **Independencia** | Los errores son independientes entre sí | Durbin-Watson |
| **Homoscedasticidad** | Varianza constante de los errores | Breusch-Pagan |
| **Normalidad** | Los residuos siguen distribución normal | Shapiro-Wilk, Q-Q plot |
| **No multicolinealidad** | Las variables predictoras no están altamente correlacionadas | VIF |

## Función de Costo: Error Cuadrático Medio (MSE)

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (Y_i - \hat{Y}_i)^2$$

El modelo de regresión busca **minimizar** esta función de costo.

---
# 2. Regresión Lineal Simple

Modelaremos la relación entre una sola variable predictora y la variable objetivo.

**Ejemplo:** ¿Podemos predecir la **Conductividad** del agua a partir de los **Sólidos disueltos**?

In [ ]:
# ═══════════════════════════════════════════════════════════
# REGRESIÓN LINEAL SIMPLE: Solids → Conductivity
# ═══════════════════════════════════════════════════════════
print("📈 REGRESIÓN LINEAL SIMPLE")
print("Variable predictora (X): Solids (Sólidos disueltos)")
print("Variable objetivo   (Y): Conductivity (Conductividad)")
print("=" * 55)

# Definir variables
X_simple = df[['Solids']].values
y_simple = df['Conductivity'].values

# Dividir en conjuntos de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

print(f"\nConjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba:        {X_test.shape[0]} muestras")

In [ ]:
# Entrenar el modelo de Regresión Lineal Simple
modelo_simple = LinearRegression()
modelo_simple.fit(X_train, y_train)

# Coeficientes del modelo
print("📋 COEFICIENTES DEL MODELO")
print("=" * 50)
print(f"  Intercepto (β₀): {modelo_simple.intercept_:.4f}")
print(f"  Pendiente  (β₁): {modelo_simple.coef_[0]:.6f}")
print(f"\n  Ecuación: Conductivity = {modelo_simple.intercept_:.4f} + {modelo_simple.coef_[0]:.6f} × Solids")

# Predicciones
y_pred_train = modelo_simple.predict(X_train)
y_pred_test = modelo_simple.predict(X_test)

In [ ]:
# Visualización de la regresión lineal simple
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Datos + Recta de regresión (Entrenamiento)
axes[0].scatter(X_train, y_train, alpha=0.3, s=15, color='steelblue', label='Datos de entrenamiento')
# Ordenar para dibujar la línea
idx_sorted = np.argsort(X_train.flatten())
axes[0].plot(X_train[idx_sorted], y_pred_train[idx_sorted], color='red', linewidth=2, label='Recta de regresión')
axes[0].set_xlabel('Sólidos Disueltos (ppm)', fontsize=12)
axes[0].set_ylabel('Conductividad (μS/cm)', fontsize=12)
axes[0].set_title('Regresión Lineal Simple\n(Datos de Entrenamiento)', fontweight='bold')
axes[0].legend()

# Gráfico 2: Valores reales vs predichos (Prueba)
axes[1].scatter(y_test, y_pred_test, alpha=0.3, s=15, color='coral')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='Línea ideal (y=x)')
axes[1].set_xlabel('Valores Reales', fontsize=12)
axes[1].set_ylabel('Valores Predichos', fontsize=12)
axes[1].set_title('Valores Reales vs Predichos\n(Datos de Prueba)', fontweight='bold')
axes[1].legend()

plt.suptitle('Regresión Lineal Simple: Sólidos → Conductividad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 3. Evaluación del Modelo Simple

## Métricas de Evaluación

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **R² (Coeficiente de Determinación)** | $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$ | % de varianza explicada (0 a 1, más cercano a 1 = mejor) |
| **MSE (Error Cuadrático Medio)** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Error promedio al cuadrado (penaliza errores grandes) |
| **RMSE (Raíz del MSE)** | $\sqrt{MSE}$ | Error en las mismas unidades que Y |
| **MAE (Error Absoluto Medio)** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Error promedio absoluto (más robusto a outliers) |

In [ ]:
# ═══════════════════════════════════════════════════════════
# MÉTRICAS DE EVALUACIÓN — MODELO SIMPLE
# ═══════════════════════════════════════════════════════════
print("📊 EVALUACIÓN DEL MODELO DE REGRESIÓN SIMPLE")
print("=" * 55)

# Métricas en entrenamiento
r2_train = r2_score(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
mae_train = mean_absolute_error(y_train, y_pred_train)

# Métricas en prueba
r2_test = r2_score(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test = mean_absolute_error(y_test, y_pred_test)

metricas_simple = pd.DataFrame({
    'Métrica': ['R²', 'MSE', 'RMSE', 'MAE'],
    'Entrenamiento': [r2_train, mse_train, rmse_train, mae_train],
    'Prueba': [r2_test, mse_test, rmse_test, mae_test]
})
metricas_simple['Entrenamiento'] = metricas_simple['Entrenamiento'].round(4)
metricas_simple['Prueba'] = metricas_simple['Prueba'].round(4)

display(metricas_simple)

print(f"\n💡 El modelo explica el {r2_test*100:.2f}% de la varianza en los datos de prueba")
print(f"   Error promedio de predicción (RMSE): {rmse_test:.2f} μS/cm")

In [ ]:
# Interpretación del coeficiente
print("📝 INTERPRETACIÓN DEL MODELO")
print("=" * 55)
print(f"\nEcuación: Conductividad = {modelo_simple.intercept_:.4f} + {modelo_simple.coef_[0]:.6f} × Sólidos")
print(f"\n• Intercepto ({modelo_simple.intercept_:.4f}): Conductividad estimada cuando los sólidos disueltos son 0")
print(f"• Pendiente ({modelo_simple.coef_[0]:.6f}): Por cada unidad (ppm) de aumento en sólidos,")
print(f"  la conductividad cambia en {modelo_simple.coef_[0]:.6f} μS/cm")
print(f"• R² = {r2_test:.4f}: El modelo explica solo el {r2_test*100:.2f}% de la variabilidad")

---
# 4. Regresión Lineal Múltiple

Ahora usaremos **múltiples variables predictoras** para mejorar la capacidad predictiva del modelo.

**Objetivo:** Predecir la **Conductividad** del agua usando todas las variables disponibles.

In [ ]:
# ═══════════════════════════════════════════════════════════
# SELECCIÓN DE FEATURES
# ═══════════════════════════════════════════════════════════
print("🔍 SELECCIÓN DE VARIABLES PREDICTORAS")
print("=" * 55)

# Variable objetivo
target = 'Conductivity'
features = [c for c in cols_numericas if c != target]

print(f"Variable objetivo: {target}")
print(f"Variables predictoras: {features}")

# Correlación de cada feature con el target
print(f"\nCorrelación con {target}:")
corr_target = df[features + [target]].corr()[target].drop(target).sort_values(ascending=False)
for feat, corr in corr_target.items():
    print(f"  {feat:25s} → r = {corr:+.4f}")

In [ ]:
# Verificar multicolinealidad antes de construir el modelo
print("📐 VERIFICACIÓN DE MULTICOLINEALIDAD (VIF)")
print("=" * 55)

X_vif = df[features].copy()
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['Variable'] = features
vif_data['VIF'] = [variance_inflation_factor(X_vif_const.values, i+1) for i in range(len(features))]
vif_data = vif_data.sort_values('VIF', ascending=False)

display(vif_data)

# Seleccionar features con VIF aceptable (< 10)
features_ok = vif_data[vif_data['VIF'] < 10]['Variable'].tolist()
print(f"\n✅ Features seleccionadas (VIF < 10): {features_ok}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# REGRESIÓN LINEAL MÚLTIPLE
# ═══════════════════════════════════════════════════════════
print("📈 REGRESIÓN LINEAL MÚLTIPLE")
print("=" * 55)

# Preparar datos
X_multi = df[features_ok].values
y_multi = df[target].values

# División entrenamiento / prueba
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

# Entrenar el modelo
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train_m, y_train_m)

# Coeficientes
print(f"\nIntercepto (β₀): {modelo_multiple.intercept_:.4f}")
print(f"\nCoeficientes:")
for feat, coef in zip(features_ok, modelo_multiple.coef_):
    print(f"  {feat:25s} → β = {coef:+.6f}")

# Predicciones
y_pred_train_m = modelo_multiple.predict(X_train_m)
y_pred_test_m = modelo_multiple.predict(X_test_m)

In [ ]:
# ═══════════════════════════════════════════════════════════
# EVALUACIÓN DEL MODELO MÚLTIPLE
# ═══════════════════════════════════════════════════════════
print("📊 EVALUACIÓN DEL MODELO MÚLTIPLE")
print("=" * 55)

# Métricas
r2_train_m = r2_score(y_train_m, y_pred_train_m)
r2_test_m = r2_score(y_test_m, y_pred_test_m)
mse_test_m = mean_squared_error(y_test_m, y_pred_test_m)
rmse_test_m = np.sqrt(mse_test_m)
mae_test_m = mean_absolute_error(y_test_m, y_pred_test_m)

# R² ajustado
n = X_test_m.shape[0]
p = X_test_m.shape[1]
r2_adj = 1 - (1 - r2_test_m) * (n - 1) / (n - p - 1)

metricas_multiple = pd.DataFrame({
    'Métrica': ['R²', 'R² Ajustado', 'MSE', 'RMSE', 'MAE'],
    'Entrenamiento': [r2_train_m, '-', mean_squared_error(y_train_m, y_pred_train_m),
                      np.sqrt(mean_squared_error(y_train_m, y_pred_train_m)),
                      mean_absolute_error(y_train_m, y_pred_train_m)],
    'Prueba': [r2_test_m, r2_adj, mse_test_m, rmse_test_m, mae_test_m]
})

display(metricas_multiple.round(4))

print(f"\n💡 El modelo múltiple explica el {r2_test_m*100:.2f}% de la varianza")
print(f"   R² ajustado (penaliza por cantidad de variables): {r2_adj:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPARACIÓN: MODELO SIMPLE vs MODELO MÚLTIPLE
# ═══════════════════════════════════════════════════════════
print("⚖️ COMPARACIÓN DE MODELOS")
print("=" * 55)

comparacion = pd.DataFrame({
    'Modelo': ['Regresión Simple (Solids)', f'Regresión Múltiple ({len(features_ok)} vars)'],
    'R² (Test)': [r2_test, r2_test_m],
    'RMSE (Test)': [rmse_test, rmse_test_m],
    'MAE (Test)': [mae_test, mae_test_m]
})

display(comparacion.round(4))

mejora_r2 = (r2_test_m - r2_test) / abs(r2_test) * 100 if r2_test != 0 else float('inf')
print(f"\n📈 Mejora en R² al pasar de simple a múltiple: {mejora_r2:+.2f}%")

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Modelo Simple: Real vs Predicho
axes[0].scatter(y_test, y_pred_test, alpha=0.3, s=15, color='steelblue')
lim_min = min(y_test.min(), y_pred_test.min())
lim_max = max(y_test.max(), y_pred_test.max())
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=1)
axes[0].set_xlabel('Valores Reales')
axes[0].set_ylabel('Valores Predichos')
axes[0].set_title(f'Regresión Simple\nR² = {r2_test:.4f}, RMSE = {rmse_test:.2f}', fontweight='bold')

# Modelo Múltiple: Real vs Predicho
axes[1].scatter(y_test_m, y_pred_test_m, alpha=0.3, s=15, color='coral')
lim_min_m = min(y_test_m.min(), y_pred_test_m.min())
lim_max_m = max(y_test_m.max(), y_pred_test_m.max())
axes[1].plot([lim_min_m, lim_max_m], [lim_min_m, lim_max_m], 'r--', linewidth=1)
axes[1].set_xlabel('Valores Reales')
axes[1].set_ylabel('Valores Predichos')
axes[1].set_title(f'Regresión Múltiple\nR² = {r2_test_m:.4f}, RMSE = {rmse_test_m:.2f}', fontweight='bold')

plt.suptitle('Comparación: Real vs Predicho', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 5. Análisis Detallado con Statsmodels

Statsmodels proporciona un resumen estadístico completo del modelo, incluyendo p-valores, intervalos de confianza y pruebas diagnósticas.

In [ ]:
# Modelo con statsmodels para obtener resumen detallado
X_sm = sm.add_constant(df[features_ok])
modelo_sm = sm.OLS(df[target], X_sm).fit()

print(modelo_sm.summary())

In [ ]:
# Interpretación de los coeficientes significativos
print("📝 INTERPRETACIÓN DE COEFICIENTES")
print("=" * 55)

coefs = pd.DataFrame({
    'Variable': modelo_sm.params.index[1:],  # Excluir constante
    'Coeficiente': modelo_sm.params.values[1:],
    'p-valor': modelo_sm.pvalues.values[1:],
    'Significativo': ['✅ Sí' if p < 0.05 else '❌ No' for p in modelo_sm.pvalues.values[1:]]
})
coefs = coefs.sort_values('p-valor')
display(coefs.round(6))

sig_vars = coefs[coefs['p-valor'] < 0.05]['Variable'].tolist()
print(f"\n✅ Variables significativas (p < 0.05): {sig_vars}")

---
# 6. Validación de Supuestos

Verificaremos los supuestos de la regresión lineal para nuestro modelo múltiple.

In [ ]:
# ═══════════════════════════════════════════════════════════
# SUPUESTO 1: LINEALIDAD
# ═══════════════════════════════════════════════════════════
print("🔍 SUPUESTO 1: LINEALIDAD")
print("=" * 50)

residuos = modelo_sm.resid
valores_predichos = modelo_sm.fittedvalues

plt.figure(figsize=(10, 5))
plt.scatter(valores_predichos, residuos, alpha=0.3, s=10, color='steelblue')
plt.axhline(y=0, color='red', linewidth=1)
plt.xlabel('Valores Predichos', fontsize=12)
plt.ylabel('Residuos', fontsize=12)
plt.title('Residuos vs Valores Predichos\n(Verificación de Linealidad)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Si los residuos están dispersos aleatoriamente alrededor de 0, se cumple la linealidad.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SUPUESTO 2: NORMALIDAD DE RESIDUOS
# ═══════════════════════════════════════════════════════════
print("🔍 SUPUESTO 2: NORMALIDAD DE RESIDUOS")
print("=" * 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de residuos con curva normal
axes[0].hist(residuos, bins=40, density=True, color='steelblue', edgecolor='white', alpha=0.7)
x_norm = np.linspace(residuos.min(), residuos.max(), 100)
axes[0].plot(x_norm, stats.norm.pdf(x_norm, residuos.mean(), residuos.std()),
             'r-', linewidth=2, label='Distribución Normal')
axes[0].set_xlabel('Residuos')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de Residuos', fontweight='bold')
axes[0].legend()

# Q-Q Plot
stats.probplot(residuos, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot de Residuos', fontweight='bold')
axes[1].get_lines()[1].set_color('red')

plt.suptitle('Verificación de Normalidad de Residuos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Prueba de Shapiro-Wilk (usar muestra si el dataset es grande)
muestra_residuos = np.random.choice(residuos, size=min(5000, len(residuos)), replace=False)
sw_stat, sw_pvalue = stats.shapiro(muestra_residuos)
print(f"\nPrueba de Shapiro-Wilk:")
print(f"  Estadístico: {sw_stat:.4f}")
print(f"  p-valor:     {sw_pvalue:.6f}")
print(f"  Conclusión:  {'✅ Normalidad (no se rechaza H₀)' if sw_pvalue >= 0.05 else '⚠️ No normalidad (se rechaza H₀)'}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SUPUESTO 3: HOMOSCEDASTICIDAD
# ═══════════════════════════════════════════════════════════
print("🔍 SUPUESTO 3: HOMOSCEDASTICIDAD")
print("=" * 50)

bp_stat, bp_pvalue, _, _ = het_breuschpagan(residuos, X_sm)

print(f"Prueba de Breusch-Pagan:")
print(f"  Estadístico LM: {bp_stat:.4f}")
print(f"  p-valor:         {bp_pvalue:.6f}")
print(f"  Conclusión:      {'❌ Heteroscedasticidad' if bp_pvalue < 0.05 else '✅ Homoscedasticidad'}")

# Visualización: residuos al cuadrado vs predichos
plt.figure(figsize=(10, 5))
plt.scatter(valores_predichos, np.sqrt(np.abs(residuos)), alpha=0.3, s=10, color='coral')
plt.xlabel('Valores Predichos', fontsize=12)
plt.ylabel('√|Residuos|', fontsize=12)
plt.title(f'Scale-Location Plot\n(Breusch-Pagan p={bp_pvalue:.4f})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# SUPUESTO 4: INDEPENDENCIA DE ERRORES
# ═══════════════════════════════════════════════════════════
print("🔍 SUPUESTO 4: INDEPENDENCIA DE ERRORES")
print("=" * 50)

# Prueba de Durbin-Watson
from statsmodels.stats.stattools import durbin_watson

dw = durbin_watson(residuos)
print(f"Prueba de Durbin-Watson:")
print(f"  Estadístico DW: {dw:.4f}")
print(f"  Interpretación:")
if dw < 1.5:
    print(f"  ⚠️ Posible autocorrelación positiva (DW < 1.5)")
elif dw > 2.5:
    print(f"  ⚠️ Posible autocorrelación negativa (DW > 2.5)")
else:
    print(f"  ✅ No hay autocorrelación significativa (1.5 < DW < 2.5)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# RESUMEN DE VALIDACIÓN DE SUPUESTOS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📋 RESUMEN DE VALIDACIÓN DE SUPUESTOS")
print("=" * 60)

supuestos = [
    ['Linealidad', 'Visual (Residuos vs Predichos)', 'Verificar gráfico'],
    ['Normalidad', f'Shapiro-Wilk (p={sw_pvalue:.4f})', '✅ Sí' if sw_pvalue >= 0.05 else '⚠️ No'],
    ['Homoscedasticidad', f'Breusch-Pagan (p={bp_pvalue:.4f})', '✅ Sí' if bp_pvalue >= 0.05 else '⚠️ No'],
    ['Independencia', f'Durbin-Watson (DW={dw:.4f})', '✅ Sí' if 1.5 <= dw <= 2.5 else '⚠️ No'],
    ['No multicolinealidad', f'VIF (máx={vif_data["VIF"].max():.2f})', '✅ Sí' if vif_data['VIF'].max() < 10 else '⚠️ No']
]

df_supuestos = pd.DataFrame(supuestos, columns=['Supuesto', 'Prueba', '¿Se cumple?'])
display(df_supuestos)

---
# 7. Caso Práctico Completo: Pipeline de Regresión

Construiremos un **pipeline completo** desde la carga del dataset hasta el modelo final, integrando todo lo aprendido en las 4 semanas.

In [ ]:
# ═══════════════════════════════════════════════════════════
# PIPELINE COMPLETO: PREDICCIÓN DE CONDUCTIVIDAD DEL AGUA
# ═══════════════════════════════════════════════════════════
print("🚀 PIPELINE COMPLETO DE REGRESIÓN LINEAL")
print("=" * 60)

# ── PASO 1: Carga de datos ──
print("\n📥 PASO 1: Carga de datos")
df_pipe = pd.read_csv(url)
print(f"   Datos cargados: {df_pipe.shape}")

# ── PASO 2: Limpieza ──
print("\n🧹 PASO 2: Limpieza de datos")
# Imputar nulos con mediana
for col in df_pipe.columns:
    nulos = df_pipe[col].isnull().sum()
    if nulos > 0:
        df_pipe[col].fillna(df_pipe[col].median(), inplace=True)
        print(f"   {col}: {nulos} nulos imputados con mediana")

# Eliminar duplicados
dupl = df_pipe.duplicated().sum()
df_pipe = df_pipe.drop_duplicates().reset_index(drop=True)
print(f"   Duplicados eliminados: {dupl}")

# ── PASO 3: Feature Engineering ──
print("\n🔧 PASO 3: Ingeniería de Características")
df_pipe['ratio_solidos_cond'] = df_pipe['Solids'] / df_pipe['Conductivity']
df_pipe['ph_neutral'] = ((df_pipe['ph'] >= 6.5) & (df_pipe['ph'] <= 8.5)).astype(int)
df_pipe['log_solids'] = np.log1p(df_pipe['Solids'])
print("   3 nuevas características creadas")

# ── PASO 4: Selección de variables ──
print("\n🎯 PASO 4: Selección de variables")
target_pipe = 'Conductivity'
features_pipe = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
                  'Organic_carbon', 'Trihalomethanes', 'Turbidity',
                  'ratio_solidos_cond', 'log_solids']
print(f"   Target: {target_pipe}")
print(f"   Features: {len(features_pipe)} variables")

# ── PASO 5: División de datos ──
print("\n✂️ PASO 5: División Train/Test")
X_pipe = df_pipe[features_pipe].values
y_pipe = df_pipe[target_pipe].values

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pipe, y_pipe, test_size=0.2, random_state=42
)
print(f"   Train: {X_train_p.shape[0]} | Test: {X_test_p.shape[0]}")

# ── PASO 6: Escalado ──
print("\n📏 PASO 6: Estandarización de features")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_p)
X_test_scaled = scaler.transform(X_test_p)
print("   ✅ Features estandarizadas (media=0, std=1)")

# ── PASO 7: Entrenamiento ──
print("\n🏋️ PASO 7: Entrenamiento del modelo")
modelo_final = LinearRegression()
modelo_final.fit(X_train_scaled, y_train_p)
print("   ✅ Modelo entrenado")

# ── PASO 8: Evaluación ──
print("\n📊 PASO 8: Evaluación")
y_pred_final = modelo_final.predict(X_test_scaled)

r2_final = r2_score(y_test_p, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test_p, y_pred_final))
mae_final = mean_absolute_error(y_test_p, y_pred_final)

n_final = X_test_scaled.shape[0]
p_final = X_test_scaled.shape[1]
r2_adj_final = 1 - (1 - r2_final) * (n_final - 1) / (n_final - p_final - 1)

print(f"   R²:          {r2_final:.4f}")
print(f"   R² Ajustado: {r2_adj_final:.4f}")
print(f"   RMSE:        {rmse_final:.4f}")
print(f"   MAE:         {mae_final:.4f}")

In [ ]:
# Visualización final del pipeline
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Real vs Predicho
axes[0, 0].scatter(y_test_p, y_pred_final, alpha=0.3, s=15, color='steelblue')
lim = [min(y_test_p.min(), y_pred_final.min()), max(y_test_p.max(), y_pred_final.max())]
axes[0, 0].plot(lim, lim, 'r--', linewidth=1)
axes[0, 0].set_xlabel('Valores Reales')
axes[0, 0].set_ylabel('Valores Predichos')
axes[0, 0].set_title(f'Real vs Predicho (R²={r2_final:.4f})', fontweight='bold')

# 2. Distribución de residuos
residuos_final = y_test_p - y_pred_final
axes[0, 1].hist(residuos_final, bins=35, color='coral', edgecolor='white', density=True)
x_norm = np.linspace(residuos_final.min(), residuos_final.max(), 100)
axes[0, 1].plot(x_norm, stats.norm.pdf(x_norm, residuos_final.mean(), residuos_final.std()),
                'k-', linewidth=2)
axes[0, 1].set_xlabel('Residuos')
axes[0, 1].set_title('Distribución de Residuos', fontweight='bold')

# 3. Residuos vs Predichos
axes[1, 0].scatter(y_pred_final, residuos_final, alpha=0.3, s=15, color='mediumseagreen')
axes[1, 0].axhline(0, color='red', linewidth=1)
axes[1, 0].set_xlabel('Valores Predichos')
axes[1, 0].set_ylabel('Residuos')
axes[1, 0].set_title('Residuos vs Predichos', fontweight='bold')

# 4. Importancia de features (coeficientes absolutos)
importancia = pd.Series(np.abs(modelo_final.coef_), index=features_pipe).sort_values()
importancia.plot(kind='barh', color='steelblue', edgecolor='white', ax=axes[1, 1])
axes[1, 1].set_xlabel('|Coeficiente| (estandarizado)')
axes[1, 1].set_title('Importancia de Variables', fontweight='bold')

plt.suptitle('Pipeline Completo — Regresión Lineal Múltiple\nPredicción de Conductividad del Agua',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo de predicción con datos nuevos
print("🔮 EJEMPLO DE PREDICCIÓN CON DATOS NUEVOS")
print("=" * 55)

# Simular una nueva muestra de agua
nueva_muestra = {
    'ph': 7.0,
    'Hardness': 200.0,
    'Solids': 20000.0,
    'Chloramines': 7.0,
    'Sulfate': 330.0,
    'Organic_carbon': 14.0,
    'Trihalomethanes': 65.0,
    'Turbidity': 4.0,
    'ratio_solidos_cond': 20000.0 / 400.0,  # Estimación
    'log_solids': np.log1p(20000.0)
}

print("Valores de la nueva muestra:")
for k, v in nueva_muestra.items():
    print(f"  {k:25s} = {v:.2f}")

# Preparar y predecir
X_nuevo = pd.DataFrame([nueva_muestra])[features_pipe].values
X_nuevo_scaled = scaler.transform(X_nuevo)
prediccion = modelo_final.predict(X_nuevo_scaled)

print(f"\n📊 Conductividad predicha: {prediccion[0]:.2f} μS/cm")

---
## Conclusiones del Curso

A lo largo de las 4 semanas, construimos un pipeline completo de análisis de datos ambientales:

### Semana 1 — Carga y Visualización
- Cargamos datos desde CSV, Excel, GitHub y Google Sheets
- Exploramos el dataset con pandas
- Visualizamos con Matplotlib, Seaborn y Plotly Express

### Semana 2 — Calidad y Transformación
- Diagnosticamos problemas de calidad
- Limpiamos datos: nulos, duplicados y outliers
- Transformamos variables: normalización, encoding, discretización
- Creamos nuevas características

### Semana 3 — Análisis Estadístico
- Analizamos correlaciones (Pearson, Spearman, Kendall)
- Evaluamos homoscedasticidad/heteroscedasticidad
- Detectamos multicolinealidad con VIF
- Formulamos y probamos hipótesis estadísticas

### Semana 4 — Regresión Lineal
- Implementamos regresión simple y múltiple
- Evaluamos con R², MSE, RMSE, MAE
- Validamos los 5 supuestos del modelo
- Construimos un pipeline completo de predicción

### Lecciones Aprendidas
1. La **preparación de datos** (limpieza, transformación) es fundamental para obtener buenos modelos
2. Siempre debemos **validar los supuestos** antes de interpretar los resultados
3. Un R² bajo no significa un modelo inútil — puede indicar que las variables disponibles no capturan toda la variabilidad del fenómeno
4. La **ingeniería de características** puede mejorar significativamente el rendimiento del modelo
5. Trabajar con datos ambientales reales implica lidiar con ruido, valores atípicos y relaciones complejas